In [2]:
# 1. code
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ["WANDB_API_KEY"] = s.get_secret("WANDB_API_KEY").strip()

!git clone -b feat/trainer-esraa https://github.com/esraa1000/multimodal-bird-vision.git project
%cd project
!git log --oneline -1
!pip install -q -r requirements.txt

Cloning into 'project'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 221 (delta 86), reused 189 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 9.64 MiB | 31.96 MiB/s, done.
Resolving deltas: 100% (86/86), done.
/kaggle/working/project
d233d75 (HEAD -> feat/trainer-esraa, origin/feat/trainer-esraa) changed branch name


In [3]:
# 2. data (goes to /kaggle/working/data automatically)
!python -m src.cli prepare-data

[prepare-data] train: downloading Multimodal-Fatima/CUB_train ...
README.md: 100%|███████████████████████████| 7.16k/7.16k [00:00<00:00, 18.2MB/s]

data/train-00000-of-00002-5e7d350dbc117c(…): downloading bytes: ▊|  247MB, 20.6M
data/train-00000-of-00002-5e7d350dbc117c(…): downloading bytes: █|  285MB, 26.2M
data/train-00000-of-00002-5e7d350dbc117c(…): reconstructing file: 100%|█|  287MB

data/train-00001-of-00002-ae08e95fea856c(…): downloading bytes: ▉|  271MB, 23.3M
data/train-00001-of-00002-ae08e95fea856c(…): reconstructing file:  45%|▍|  134MB
data/train-00001-of-00002-ae08e95fea856c(…): downloading bytes: █|  295MB, 22.5M
data/train-00001-of-00002-ae08e95fea856c(…): reconstructing file: 100%|█|  297MB
Generating train split: 100%|██████| 5994/5994 [00:03<00:00, 1591.78 examples/s]
Saving the dataset (2/2 shards): 100%|█| 5994/5994 [00:02<00:00, 2724.37 example
[prepare-data] train: 5994 records -> /content/data/cub/train
[prepare-data] test: downloading Multimodal-Fatima/CUB_test 

In [5]:
%%bash
CUDA_VISIBLE_DEVICES=0 python -m src.cli tune --config configs/tune_grid.yaml   --shard 1/4 data.num_workers=2 > grid.log   2>&1 &
wait

Process is interrupted.


In [6]:
!pkill -f "src.cli tune"

In [7]:
!nvidia-smi

Mon Oct 26 10:50:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!cat /kaggle/working/project/outputs/tune_grid_v1/results.jsonl 2>/dev/null; cat /kaggle/working/project/outputs/tune_grid_v2/results.jsonl

{"tune": "tune_grid_v2", "method": "grid", "trial_id": 0, "params": {"train.lr": 0.0003, "train.weight_decay": 0.01, "model.dropout": 0.1}, "status": "ok", "score": 0.24707846410684475, "best_epoch": 14, "time_s": 810.3, "gpu": "Tesla T4"}
{"tune": "tune_grid_v2", "method": "grid", "trial_id": 0, "params": {"train.lr": 0.0003, "train.weight_decay": 0.01, "model.dropout": 0.1}, "status": "ok", "score": 0.23205342237061768, "best_epoch": 14, "time_s": 984.1, "gpu": "Tesla T4"}
{"tune": "tune_grid_v2", "method": "grid", "trial_id": 4, "params": {"train.lr": 0.0003, "train.weight_decay": 0.1, "model.dropout": 0.3}, "status": "ok", "score": 0.21869782971619364, "best_epoch": 14, "time_s": 918.0, "gpu": "Tesla T4"}


In [9]:
%%bash
CUDA_VISIBLE_DEVICES=0 python -m src.cli tune --config configs/tune_grid.yaml --shard 1/2 data.num_workers=2 > g1.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python -m src.cli tune --config configs/tune_grid.yaml --shard 2/2 data.num_workers=2 > g2.log 2>&1 &
wait

/bin/bash: line 1: nividia-smi: command not found


In [18]:
%%bash
CUDA_VISIBLE_DEVICES=0 python -m src.cli tune --config configs/tune_random.yaml --shard 1/8 data.num_workers=2 > r1.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python -m src.cli tune --config configs/tune_random.yaml --shard 5/8 data.num_workers=2 > r2.log 2>&1 &
wait

In [19]:
%%bash
CUDA_VISIBLE_DEVICES=0 python -m src.cli tune --config configs/tune_random.yaml --shard 1/2 data.num_workers=2 > r1.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python -m src.cli tune --config configs/tune_random.yaml --shard 2/2 data.num_workers=2 > r2.log 2>&1 &
wait

got best params from both gridsearch and random search and will train the encoder again using them

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python -m src.cli train --config configs/cls_scratch_residual.yaml train.lr=0.002667 train.weight_decay=0.02088 model.dropout=0.173 run_name=scratch_tuned_random data.num_workers=2 > t1.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python -m src.cli train --config configs/cls_scratch_residual.yaml train.lr=0.001 train.weight_decay=0.01 model.dropout=0.1 run_name=scratch_tuned_grid data.num_workers=2 > t2.log 2>&1 &
wait

In [1]:
git add reports/tuning
git commit -m "Tuning report: grid vs random search (v2, residual)"
git push

SyntaxError: invalid syntax (492788865.py, line 1)

In [6]:
import wandb
art = wandb.Api().artifact("deep_learning_birds/multimodal-bird-vision/40qg3kcq-best_model:v0")
ckpt_dir = art.download("/kaggle/working/ckpt")
print(ckpt_dir)

wandb:   1 of 1 files downloaded.  


/kaggle/working/ckpt


In [7]:
!python -m src.cli evaluate --config configs/cls_scratch_residual.yaml --ckpt /kaggle/working/ckpt/best.pt --split test run_name=scratch_final

[local tracker] run 'scratch_final_eval-test' -> /kaggle/working/project/outputs/scratch_final_eval-test_20261026-170757
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: esraaaboelkhair8 (deep_learning_birds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/project/outputs/scratch_final_eval-test_20261026-170757/wandb/run-20261026_170800-h6ynwddv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scratch_final_eval-test
wandb: ⭐️ View project at https://wandb.ai/deep_learning_birds/multimodal-bird-vision
wandb: 🚀 View run at https://wandb.ai/deep_learning_birds/multimodal-bird-v

one final trail with 80 epochs and strong augmentation to prevent overfitting

In [8]:
!python -m src.cli train --config configs/cls_scratch_residual.yaml data.augment=strong train.epochs=80 run_name=scratch_residual_strong80

[local tracker] run 'scratch_residual_strong80' -> /kaggle/working/project/outputs/scratch_residual_strong80_20261026-171620
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: esraaaboelkhair8 (deep_learning_birds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/project/outputs/scratch_residual_strong80_20261026-171620/wandb/run-20261026_171622-d4zbwocv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scratch_residual_strong80
wandb: ⭐️ View project at https://wandb.ai/deep_learning_birds/multimodal-bird-vision
wandb: 🚀 View run at https://wandb.ai/deep_learning_birds/multimodal-bird-vision/runs/d4zbwocv
[trainer]

In [9]:
import glob
ckpt = sorted(glob.glob("/kaggle/working/project/outputs/scratch_residual_strong80_*/best.pt"))[-1]
print(ckpt)
!python -m src.cli evaluate --config configs/cls_scratch_residual.yaml --ckpt {ckpt} --split val run_name=scratch_residual_strong80

/kaggle/working/project/outputs/scratch_residual_strong80_20261026-171620/best.pt
[local tracker] run 'scratch_residual_strong80_eval-val' -> /kaggle/working/project/outputs/scratch_residual_strong80_eval-val_20261026-180946
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: esraaaboelkhair8 (deep_learning_birds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/project/outputs/scratch_residual_strong80_eval-val_20261026-180946/wandb/run-20261026_180947-kuiotsgh
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scratch_residual_strong80_eval-val
wandb: ⭐️ View project at https://wandb.ai/deep_learning_birds/multimoda

In [10]:
!python -m src.cli evaluate --config configs/cls_scratch_residual.yaml --ckpt {ckpt} --split test run_name=scratch_residual_strong80_final

[local tracker] run 'scratch_residual_strong80_final_eval-test' -> /kaggle/working/project/outputs/scratch_residual_strong80_final_eval-test_20261026-181051
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: esraaaboelkhair8 (deep_learning_birds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/project/outputs/scratch_residual_strong80_final_eval-test_20261026-181051/wandb/run-20261026_181053-99s3e1dq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scratch_residual_strong80_final_eval-test
wandb: ⭐️ View project at https://wandb.ai/deep_learning_birds/multimodal-bird-vision
wandb: 🚀 View run at https://wandb.ai/de

one last run with light augmentation for 80 epochs 

In [11]:
!python -m src.cli train --config configs/cls_scratch_residual.yaml data.augment=light train.epochs=80 run_name=scratch_residual_light80

[local tracker] run 'scratch_residual_light80' -> /kaggle/working/project/outputs/scratch_residual_light80_20261026-182119
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: esraaaboelkhair8 (deep_learning_birds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/project/outputs/scratch_residual_light80_20261026-182119/wandb/run-20261026_182120-4irq7fkj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scratch_residual_light80
wandb: ⭐️ View project at https://wandb.ai/deep_learning_birds/multimodal-bird-vision
wandb: 🚀 View run at https://wandb.ai/deep_learning_birds/multimodal-bird-vision/runs/4irq7fkj
[trainer] 11.

In [12]:
import glob
ckpt = sorted(glob.glob("/kaggle/working/project/outputs/scratch_residual_light80_*/best.pt"))[-1]
print(ckpt)
!python -m src.cli evaluate --config configs/cls_scratch_residual.yaml --ckpt {ckpt} --split val run_name=scratch_residual_light80

/kaggle/working/project/outputs/scratch_residual_light80_20261026-182119/best.pt
[local tracker] run 'scratch_residual_light80_eval-val' -> /kaggle/working/project/outputs/scratch_residual_light80_eval-val_20261026-185836
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: esraaaboelkhair8 (deep_learning_birds) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 9v0quiek (0.4s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/project/outputs/scratch_residual_light80_eval-val_20261026-185836/wandb/run-20261026_185838-9v0quiek
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run scratch_residual_light80_eval-val
